# Author and test Trouves in a notebook

A Trouve is a Python object in a Python file. Thus a notebook is a good place to write
one, to read it, and to test the logic of it before a warehouse sees it.

This notebook shows four things:

1. a `PandasTrouve` transform that you call directly, with your own DataFrame
2. a `SeedTrouve` that holds its rows in the file
3. the tests, the columns, and the docs of a Trouve
4. a complete project that the notebook writes to a temporary directory, and compiles

Nothing here needs a warehouse.

In [1]:
from pathlib import Path


def find_examples_root(start: Path | None = None) -> Path:
    """Give the directory that holds examples/projects.

    The search starts at the working directory and it goes up. Thus the
    notebook runs from this directory, from the repository root, or from a
    Jupyter server that you started at a different place.
    """
    for directory in [start or Path.cwd(), *(start or Path.cwd()).parents]:
        if (directory / "examples" / "projects").is_dir():
            return directory
    raise FileNotFoundError("No parent directory holds examples/projects.")


REPOSITORY_ROOT = find_examples_root()

# The CLI configures structlog; a notebook does not. This line keeps the INFO
# messages of each operation out of the cells. Remove it to read them.
import logging

import structlog

structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import os
import sys

import pandas as pd

import clair

PROJECT = REPOSITORY_ROOT / "examples" / "projects" / "example_4"
os.environ["CLAIR_USER"] = "notebook_user"

# The Trouve files import each other by the project layout, thus the project
# root must be on the path before the notebook imports one.
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

This notebook writes its own environments file, thus it needs no `~/.clair/environments.yml`.
[01_python_api_tour.ipynb](01_python_api_tour.ipynb) explains the cell.

In [2]:
import tempfile
from pathlib import Path

import clair.environments.environments as environments_module

NOTEBOOK_DIRECTORY = Path(tempfile.mkdtemp(prefix="clair-notebook-"))
environments_file = NOTEBOOK_DIRECTORY / "environments.yml"
environments_file.write_text(
    """
dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4
""".strip()
)

# clair reads ~/.clair/environments.yml. This line points it at the file above.
environments_module.DEFAULT_ENVIRONMENTS_PATH = environments_file

print(environments_file.read_text())

dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4


## 1. A pandas transform is an ordinary function

A `PandasTrouve` holds a `transform` function. clair reads the upstream tables into
DataFrames, it calls your function on the machine that runs clair, and it writes the result
to the warehouse. The function takes DataFrames and it gives a DataFrame. Thus you call it
here with data of your own, and you see the result at once.

In [3]:
from example_4_database.derived.daily_event_counts import daily_event_counts
from example_4_database.derived.daily_event_counts import trouve as counts_trouve

sample_events = pd.DataFrame(
    {
        "event_id": ["1", "2", "3", "4", "5", "6"],
        "user_id": ["usr_abc", "usr_abc", "usr_def", "usr_def", "usr_ghi", "usr_ghi"],
        "event_type": [
            "page_view",
            "button_click",
            "page_view",
            "form_submit",
            "purchase",
            "purchase",
        ],
        "event_date": pd.to_datetime(
            [
                "2024-01-15",
                "2024-01-15",
                "2024-01-15",
                "2024-01-16",
                "2024-01-16",
                "2024-01-16",
            ]
        ).date,
    }
)

daily_event_counts(sample_events)

,event_date,event_type,event_count
0,2024-01-15,button_click,1
1,2024-01-15,page_view,2
2,2024-01-16,form_submit,1
3,2024-01-16,purchase,2


That call is the complete unit test of the Trouve. Put the same three lines in
`tests/` and your continuous integration tests the transformation logic with no warehouse
and no credit.

`inputs` binds the upstream Trouves to the parameters of the function **by position**. The
first Trouve of `inputs` becomes the first parameter.

In [4]:
import inspect

print("transform: ", counts_trouve.transform.__name__)
print("parameters:", list(inspect.signature(counts_trouve.transform).parameters))
print("inputs:    ", [type(input_trouve).__name__ for input_trouve in counts_trouve.inputs])
print("columns:   ", [column.name for column in counts_trouve.columns])
print("tests:     ", [test.__class__.__name__ for test in counts_trouve.tests])

transform:  daily_event_counts
parameters: ['refined_events']
inputs:     ['Trouve']
columns:    ['event_date', 'event_type', 'event_count']
tests:      ['TestUniqueColumns', 'TestNotNull']


### Compare the pandas result against the declared columns

A common fault is a transform that gives a column name that the `Column` list does not
hold. The notebook finds it before the run does.

In [5]:
result = daily_event_counts(sample_events)
declared = {column.name for column in counts_trouve.columns}
produced = set(result.columns)

print("declared, not produced:", sorted(declared - produced) or "none")
print("produced, not declared:", sorted(produced - declared) or "none")

declared, not produced: none
produced, not declared: none


## 2. A seed holds its rows in the Python file

A `SeedTrouve` is a table with the data in the file: country codes, tax rates, or a map
from an id to a label. It reads no other Trouve, thus it is always a root of the DAG.
clair builds it in the same run as each other Trouve, before the Trouves that read it.

In [6]:
from example_4_database.reference.event_type_labels import trouve as labels_trouve

labels_trouve.dataframe

,event_type,label,is_conversion
0,page_view,Page view,False
1,add_to_cart,Add to cart,False
2,purchase,Purchase,True


The dtype of each column decides the Snowflake type of that column, thus a seed sets the
dtype. Read [the seeds topic](https://rivage-sh.github.io/clair/topics/seeds/).

A seed and a pandas result join in the notebook, exactly as they do in the warehouse:

In [7]:
result.merge(labels_trouve.dataframe, on="event_type", how="left")

,event_date,event_type,event_count,label,is_conversion
0,2024-01-15,button_click,1,<NA>,NaN
1,2024-01-15,page_view,2,Page view,False
2,2024-01-16,form_submit,1,<NA>,NaN
3,2024-01-16,purchase,2,Purchase,True


## 3. Read the tests, the columns and the docs

Each Trouve carries its documentation and its data quality tests. `clair test` runs the
tests against the warehouse; the objects themselves need no connection.

In [8]:
from clair.lineage import get_dag

dag = get_dag(PROJECT)

pd.DataFrame(
    [
        {
            "trouve": address.split(".", 1)[1],
            "type": dag.get_trouve(address).type.value,
            "test": test.__class__.__name__,
            "target": getattr(test, "column", None) or getattr(test, "columns", None) or "(table)",
        }
        for address in dag.nodes
        for test in (dag.get_trouve(address).tests or [])
    ]
)

,trouve,type,test,target
0,derived.daily_event_counts,table,TestUniqueColumns,"[event_date, event_type]"
1,derived.daily_event_counts,table,TestNotNull,event_count


## 4. Write a project from the notebook, and compile it

A project is a directory tree. The directory names give the Snowflake names:
`<database>/<schema>/<table>.py` becomes `database.schema.table`. This cell writes a small
project to a temporary directory, and compiles it.

In [9]:
import tempfile
from pathlib import Path

notebook_project = Path(tempfile.mkdtemp(prefix="clair-notebook-project-")) / "shop"
(notebook_project / "shop_database" / "source").mkdir(parents=True)
(notebook_project / "shop_database" / "refined").mkdir(parents=True)
(notebook_project / "shop_database" / "derived").mkdir(parents=True)

(notebook_project / "shop_database" / "__database_config__.py").write_text(
    """
from clair import DatabaseDefaults

defaults = DatabaseDefaults(warehouse="compute_wh")
""".lstrip()
)

(notebook_project / "shop_database" / "source" / "orders.py").write_text(
    """
from clair import Column, ColumnType, Trouve, TrouveType

trouve = Trouve(
    type=TrouveType.SOURCE,
    docs="The orders that the shop application writes.",
    columns=[
        Column(name="order_id", type=ColumnType.STRING),
        Column(name="customer_id", type=ColumnType.STRING),
        Column(name="amount", type=ColumnType.FLOAT),
        Column(name="ordered_at", type=ColumnType.TIMESTAMP_NTZ),
    ],
)
""".lstrip()
)

(notebook_project / "shop_database" / "refined" / "orders.py").write_text(
    """
from shop_database.source.orders import trouve as source_orders

from clair import Column, ColumnType, TestNotNull, TestUnique, Trouve

trouve = Trouve(
    docs="One row for each order, with the date of the order.",
    sql=f\"\"\"
        select
            order_id,
            customer_id,
            amount,
            ordered_at::date as order_date
        from {source_orders}
        where amount > 0
    \"\"\",
    columns=[
        Column(name="order_id", type=ColumnType.STRING),
        Column(name="customer_id", type=ColumnType.STRING),
        Column(name="amount", type=ColumnType.FLOAT),
        Column(name="order_date", type=ColumnType.DATE),
    ],
    tests=[TestUnique(column="order_id"), TestNotNull(column="amount")],
)
""".lstrip()
)

(notebook_project / "shop_database" / "derived" / "daily_revenue.py").write_text(
    """
from shop_database.refined.orders import trouve as refined_orders

from clair import Column, ColumnType, Trouve

trouve = Trouve(
    docs="The revenue of each day.",
    sql=f\"\"\"
        select
            order_date,
            sum(amount) as revenue,
            count(*) as order_count
        from {refined_orders}
        group by order_date
    \"\"\",
    columns=[
        Column(name="order_date", type=ColumnType.DATE),
        Column(name="revenue", type=ColumnType.FLOAT),
        Column(name="order_count", type=ColumnType.NUMBER),
    ],
)
""".lstrip()
)

print(sorted(str(path.relative_to(notebook_project)) for path in notebook_project.rglob("*.py")))

['shop_database/__database_config__.py', 'shop_database/derived/daily_revenue.py', 'shop_database/refined/orders.py', 'shop_database/source/orders.py']


The project holds no `__routing__.py`. clair then writes to the logical addresses, and it
prints a warning that names the environment. Add a routing entry when two people, or two
environments, must not write to one table. Read
[the routing topic](https://rivage-sh.github.io/clair/topics/routing/).

In [10]:
output = clair.compile(notebook_project, env="prod")

for node in output.compiled_nodes:
    print(f"{node.physical_address:<44} reads {node.dependencies}")

print()
print(output.node("shop_database.derived.daily_revenue").sql[0].strip())

shop_database.refined.orders                 reads ['shop_database.source.orders']
shop_database.derived.daily_revenue          reads ['shop_database.refined.orders']

CREATE OR REPLACE TABLE shop_database.derived.daily_revenue__clair_01a04a9442c07db780763251bd663dde AS (
select
            order_date,
            sum(amount) as revenue,
            count(*) as order_count
        from shop_database.refined.orders
        group by order_date
)


The reference in the f-string became the full physical address of the parent, and clair
made the edge of the DAG from that reference. You wrote no `ref()` and no YAML.

`clair.validate()` reads the same project and finds a fault of the addresses:

In [11]:
report = clair.validate(notebook_project, env="prod")
print(f"is_valid: {report.is_valid}  routing: {report.routing_description}")

is_valid: True  routing: none


### A fault that clair finds for you

Change the SQL to name a table as text, in place of the f-string reference. The DAG then
holds no edge, thus clair can build `daily_revenue` before `refined.orders`. `validate`
reports that text reference.

In [12]:
(notebook_project / "shop_database" / "derived" / "daily_revenue.py").write_text(
    """
from clair import Column, ColumnType, Trouve

trouve = Trouve(
    docs="The revenue of each day. This SQL names the parent as text -- a fault.",
    sql=\"\"\"
        select order_date, sum(amount) as revenue
        from shop_database.refined.orders
        group by order_date
    \"\"\",
    columns=[
        Column(name="order_date", type=ColumnType.DATE),
        Column(name="revenue", type=ColumnType.FLOAT),
    ],
)
""".lstrip()
)

report = clair.validate(notebook_project, env="prod")
print(f"is_valid: {report.is_valid}")
for text_reference in report.text_references:
    print(
        f"  {text_reference.logical_address} names {text_reference.text_address}"
        f" as text, in the {text_reference.location.value}"
    )

is_valid: False
  shop_database.derived.daily_revenue names shop_database.refined.orders as text, in the sql


In [13]:
import shutil

shutil.rmtree(notebook_project.parent)
shutil.rmtree(NOTEBOOK_DIRECTORY)
clair.clean(PROJECT)
print("removed the temporary project, the environments file, and the artifacts")

removed the temporary project, the environments file, and the artifacts


## Next

- [The Trouve](https://rivage-sh.github.io/clair/topics/trouve/) — what a Trouve is, and each field of one.
- [Data quality tests](https://rivage-sh.github.io/clair/topics/data-quality-tests/) — each test, and the
  guarantee that a test gives.
- [03_run_without_snowflake.ipynb](03_run_without_snowflake.ipynb) — run this project
  against an adapter that you write.